# 03. AnthropicAdapter — deep dive

`AnthropicAdapter` is the translation layer between ArcLLM's universal
types (`Message`, `Tool`, `LLMResponse`) and the Anthropic Messages API.
It handles every quirk of the wire format so callers can stay generic:
extracting `system` to a top-level field, renaming `parameters` to
`input_schema`, mapping `role="tool"` to `role="user"` with
`tool_result` blocks, parsing `thinking` content, and surfacing usage
metadata including cache reads.

This notebook is the single-adapter deep dive. The catalog and
cross-provider comparison live in `arcllm/13-open-providers`. We do
not duplicate that material here.

You will learn:
- How `AnthropicAdapter` extends `BaseAdapter` and what each method does
- How a `list[Message]` becomes a fully-formed Anthropic request body
- How a raw Anthropic JSON response is parsed back into `LLMResponse`
- How tool calls round-trip end-to-end with a mocked HTTP client
- How streaming events look on the wire (SSE chunk types)
- How thinking models are advertised via `supports_thinking` metadata
- How errors map to `ArcLLMAPIError`, including `retry_after` parsing

## 1. Setup

In [ ]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

The live API section near the bottom is gated by `HAS_LIVE_KEY`.
Every other cell runs deterministically against a patched
`httpx.AsyncClient.post` — the entire notebook works without an
internet connection or an Anthropic key.

In [ ]:
# (live) optional — set this to run real-API sections; mock cells run regardless
HAS_LIVE_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
print(f"Live API key: {'present' if HAS_LIVE_KEY else 'missing — live cells will skip'}")

### Imports and a fixture config

We use a fake `ProviderConfig` for the mock sections so the notebook is
self-contained. The live section near the end loads the real packaged
config via `load_provider_config('anthropic')`.

In [ ]:
import json
from typing import Any
from unittest.mock import AsyncMock, patch

import httpx

from arcllm import (
    AnthropicAdapter,
    ArcLLMAPIError,
    ArcLLMConfigError,
    ArcLLMParseError,
    BaseAdapter,
    ImageBlock,
    Message,
    ModelMetadata,
    ProviderConfig,
    ProviderSettings,
    TextBlock,
    Tool,
    ToolResultBlock,
    ToolUseBlock,
    load_provider_config,
)

# Test API key — never read by Anthropic, only by the adapter at init.
os.environ["ARCLLM_TEST_KEY"] = "sk-test-walkthrough-key"

FAKE_MODEL = "claude-sonnet-4-6"

fake_config = ProviderConfig(
    provider=ProviderSettings(
        api_format="anthropic-messages",
        base_url="https://api.anthropic.com",
        api_key_env="ARCLLM_TEST_KEY",
        api_key_required=True,
        default_model=FAKE_MODEL,
        default_temperature=0.7,
    ),
    models={
        FAKE_MODEL: ModelMetadata(
            context_window=200_000,
            max_output_tokens=16_384,
            supports_tools=True,
            supports_vision=True,
            supports_thinking=True,
            input_modalities=["text", "image"],
            cost_input_per_1m=3.00,
            cost_output_per_1m=15.00,
            cost_cache_read_per_1m=0.30,
            cost_cache_write_per_1m=3.75,
        ),
    },
)
print(f"Fake config built. Default model: {fake_config.provider.default_model}")

## 2. The adapter contract

`AnthropicAdapter` extends `BaseAdapter`, which itself implements the
abstract `LLMProvider` from `arcllm.types`. The contract every adapter
must satisfy is small:

| Method | Purpose |
|---|---|
| `invoke(messages, tools, **kwargs)` | One round-trip to the provider, returning `LLMResponse` |
| `validate_config()` | True iff this adapter has the credentials it needs |
| `close()` | Release the underlying httpx client |

Everything else in `AnthropicAdapter` (`_build_headers`,
`_extract_system`, `_format_message`, `_format_tool`,
`_build_request_body`, `_parse_response`, `_parse_tool_call`,
`_parse_usage`, `_map_stop_reason`) is private plumbing — small,
testable, single-purpose.

In [ ]:
# The base class is concrete — it has real impls for everything except invoke()
adapter = AnthropicAdapter(fake_config, FAKE_MODEL)

print(f"adapter.name             = {adapter.name!r}")
print(f"adapter.model_name       = {adapter.model_name!r}")
print(f"isinstance BaseAdapter?  = {isinstance(adapter, BaseAdapter)}")
print(f"validate_config()        = {adapter.validate_config()}")
print(f"_model_meta.supports_thinking = {adapter._model_meta.supports_thinking}")

### Fail-fast key resolution

`BaseAdapter.__init__` reads the env var named in `api_key_env`. If
the variable is missing or empty and `api_key_required=True`, it
raises `ArcLLMConfigError` immediately — not three hours into a batch
job.

In [ ]:
# Missing key → ArcLLMConfigError at construction time
saved = os.environ.pop("ARCLLM_TEST_KEY")
try:
    AnthropicAdapter(fake_config, FAKE_MODEL)
except ArcLLMConfigError as e:
    print(f"got: {type(e).__name__}: {e}")
finally:
    os.environ["ARCLLM_TEST_KEY"] = saved

In [ ]:
# Operator-resolved key (e.g. from Vault) takes priority over env var
adapter_vault = AnthropicAdapter(
    fake_config,
    FAKE_MODEL,
    resolved_api_key="sk-from-vault-not-env",
)
print(f"adapter_vault._api_key = {adapter_vault._api_key!r}")
print("(env var was bypassed — Vault path keeps secrets out of os.environ)")

## 3. Building requests

`_build_request_body` is the assembly line. It walks every message,
formats every content block, extracts the system prompt to a
top-level field, and emits the JSON body httpx will POST.

In [ ]:
# Headers — Anthropic uses x-api-key, not Authorization: Bearer
adapter = AnthropicAdapter(fake_config, FAKE_MODEL)
headers = adapter._build_headers()
for k, v in headers.items():
    if k == "x-api-key":
        v = v[:7] + "…"
    print(f"  {k}: {v}")

### 3a. System extraction

The Anthropic API takes `system` as a top-level string, **not** as a
message in the array. ArcLLM keeps the universal shape (`role="system"`
in the message list) and lets the adapter peel it out at the wire.
Multiple system messages concatenate with newlines.

In [ ]:
# Single system message
sys_text, rest = adapter._extract_system(
    [
        Message(role="system", content="You are a helpful assistant."),
        Message(role="user", content="Hi"),
    ]
)
print(f"system  = {sys_text!r}")
print(f"messages = {[m.role for m in rest]}")

In [ ]:
# Multiple system messages → concatenated with \n
sys_text, rest = adapter._extract_system(
    [
        Message(role="system", content="Be concise."),
        Message(role="system", content="Use tools when needed."),
        Message(role="user", content="Hi"),
    ]
)
print(f"system  = {sys_text!r}")
print(f"messages = {[m.role for m in rest]}")

# No system message at all → None
sys_text, rest = adapter._extract_system([Message(role="user", content="Hi")])
print(f"\nno-system case: system={sys_text!r}, messages={[m.role for m in rest]}")

### 3b. Content-block formatting

Every block type has a tiny pure function. `_format_content_block`
dispatches on `isinstance` — flat, explicit, no ambiguity.

In [ ]:
blocks = [
    TextBlock(text="Hello!"),
    ImageBlock(source="iVBORw0KG…", media_type="image/png"),
    ToolUseBlock(id="toolu_01", name="search", arguments={"query": "cats"}),
    ToolResultBlock(tool_use_id="toolu_01", content="found 3 results"),
    ToolResultBlock(
        tool_use_id="toolu_02",
        content=[TextBlock(text="row 1"), TextBlock(text="row 2")],
    ),
]
for b in blocks:
    print(json.dumps(adapter._format_content_block(b), indent=2))
    print("---")

Notice three things in that output:

1. `ToolUseBlock.arguments` becomes `input` — Anthropic's name for the
   same field.
2. `ImageBlock` becomes a nested `source` object with `type=base64`.
3. `ToolResultBlock` accepts either a string or a list of nested
   blocks; the adapter walks the list recursively.

### 3c. Role translation

`role="tool"` does not exist on the Anthropic wire. Tool results are
sent as `role="user"` with `tool_result` content blocks.

In [ ]:
tool_msg = Message(
    role="tool",
    content=[ToolResultBlock(tool_use_id="t1", content="42")],
)
print(json.dumps(adapter._format_message(tool_msg), indent=2))

# user/assistant pass through
print()
print("user role passthrough:")
print(
    json.dumps(
        adapter._format_message(Message(role="user", content="hi")),
        indent=2,
    )
)

### 3d. Tool definitions

ArcLLM uses `Tool.parameters` (a JSON Schema dict). Anthropic expects
`input_schema`. Same data, different key — that's all `_format_tool`
does.

In [ ]:
tool = Tool(
    name="search_database",
    description="Search the knowledge base",
    parameters={
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Search query"},
            "limit": {"type": "integer", "default": 10},
        },
        "required": ["query"],
    },
)
print(json.dumps(adapter._format_tool(tool), indent=2))

### 3e. Full request body

`_build_request_body` glues everything: system extraction, message
formatting, tool formatting, and `_resolve_defaults` for `max_tokens`
and `temperature` (kwargs override model meta override config).

In [ ]:
body = adapter._build_request_body(
    [
        Message(role="system", content="You are a research assistant."),
        Message(role="user", content="Find papers about LLM agents"),
    ],
    tools=[tool],
)
print(json.dumps(body, indent=2))

In [ ]:
# kwargs override defaults from model metadata / config
body_default = adapter._build_request_body([Message(role="user", content="hi")])
body_override = adapter._build_request_body(
    [Message(role="user", content="hi")],
    max_tokens=512,
    temperature=0.0,
)
print(
    "default:  max_tokens=",
    body_default["max_tokens"],
    ", temperature=",
    body_default["temperature"],
)
print(
    "override: max_tokens=",
    body_override["max_tokens"],
    ", temperature=",
    body_override["temperature"],
)

In [ ]:
# tool_choice is forwarded only when both tools and tool_choice are present
body_force = adapter._build_request_body(
    [Message(role="user", content="search now")],
    tools=[tool],
    tool_choice={"type": "tool", "name": "search_database"},
)
print(json.dumps(body_force["tool_choice"], indent=2))
print()
# No tools? tool_choice is dropped silently — Anthropic would 400 otherwise.
body_no_tools = adapter._build_request_body(
    [Message(role="user", content="hi")],
    tool_choice={"type": "auto"},
)
print(f"tool_choice in body? {'tool_choice' in body_no_tools}")

## 4. Parsing responses

The Anthropic response is a JSON envelope with a `content` array of
typed blocks. `_parse_response` walks that array, splits text from
tool calls from thinking, and packages everything into the universal
`LLMResponse`.

Helper to fabricate canned responses:

In [ ]:
def make_anthropic_response(
    blocks: list[dict[str, Any]],
    *,
    stop_reason: str = "end_turn",
    input_tokens: int = 100,
    output_tokens: int = 50,
    **extra_usage: Any,
) -> dict[str, Any]:
    """Mint a fake Anthropic JSON response for parsing tests."""
    usage = {"input_tokens": input_tokens, "output_tokens": output_tokens}
    usage.update(extra_usage)
    return {
        "id": "msg_walkthrough",
        "type": "message",
        "role": "assistant",
        "model": FAKE_MODEL,
        "content": blocks,
        "stop_reason": stop_reason,
        "usage": usage,
    }


sample = make_anthropic_response([{"type": "text", "text": "Hello there."}])
print(json.dumps(sample, indent=2))

In [ ]:
# 4a. Pure text response
raw = make_anthropic_response([{"type": "text", "text": "It's sunny."}])
resp = adapter._parse_response(raw)
print(f"content      = {resp.content!r}")
print(f"tool_calls   = {resp.tool_calls}")
print(f"thinking     = {resp.thinking!r}")
print(f"stop_reason  = {resp.stop_reason!r}")
print(f"model        = {resp.model!r}")
print(f"usage        = {resp.usage}")

In [ ]:
# 4b. Tool-only response
raw = make_anthropic_response(
    [
        {
            "type": "tool_use",
            "id": "toolu_01",
            "name": "search",
            "input": {"query": "LLM agents"},
        }
    ],
    stop_reason="tool_use",
)
resp = adapter._parse_response(raw)
print(f"content      = {resp.content!r}")
print(f"stop_reason  = {resp.stop_reason}")
for tc in resp.tool_calls:
    print(f"  tool_call  = {tc}")

In [ ]:
# 4c. Mixed text + tool — model reasons aloud, then calls a tool
raw = make_anthropic_response(
    [
        {"type": "text", "text": "Let me search for that."},
        {
            "type": "tool_use",
            "id": "toolu_02",
            "name": "search",
            "input": {"query": "cats"},
        },
    ],
    stop_reason="tool_use",
)
resp = adapter._parse_response(raw)
print(f"content     = {resp.content!r}")
print(f"tool_calls  = {[tc.name for tc in resp.tool_calls]}")

### Stop reason normalization

ArcLLM normalizes Anthropic's `end_turn / tool_use / max_tokens /
stop_sequence` directly. Unknown values fall back to `end_turn` so
loops don't crash on a future provider quirk.

In [ ]:
for raw_reason in ["end_turn", "tool_use", "max_tokens", "stop_sequence", "future_unknown_reason"]:
    print(f"{raw_reason:30s} → {adapter._map_stop_reason(raw_reason)!r}")

### Usage parsing — including cache hits

Anthropic returns prompt-cache token counts on a cache-eligible call
under different keys (`cache_read_input_tokens`,
`cache_creation_input_tokens`). The adapter copies them to ArcLLM's
universal `Usage.cache_read_tokens` / `cache_write_tokens`.

In [ ]:
raw = make_anthropic_response(
    [{"type": "text", "text": "Cached!"}],
    cache_read_input_tokens=1200,
    cache_creation_input_tokens=300,
)
usage = adapter._parse_usage(raw["usage"])
print(usage)

## 5. Tool calls — full round-trip with mock

We patch `httpx.AsyncClient.post` so `invoke()` returns canned JSON
without a real network call. The flow is exactly what an agent loop
would drive: turn 1 the model asks for a tool, the harness runs the
tool, turn 2 the model gives a final answer.

In [ ]:
def fake_post(json_payload: dict[str, Any], status: int = 200) -> AsyncMock:
    """Build an AsyncMock returning a canned httpx.Response."""
    response = httpx.Response(
        status,
        json=json_payload if status == 200 else None,
        text=json.dumps(json_payload) if status != 200 else None,
        request=httpx.Request("POST", "https://api.anthropic.com/v1/messages"),
    )
    return AsyncMock(return_value=response)


# Turn 1 — model decides to call get_weather
turn1 = make_anthropic_response(
    [
        {
            "type": "tool_use",
            "id": "toolu_round_1",
            "name": "get_weather",
            "input": {"city": "Austin"},
        }
    ],
    stop_reason="tool_use",
)
print(json.dumps(turn1, indent=2)[:200] + "…")

In [ ]:
# Argument parsing edge cases — _parse_tool_call must handle dicts AND strings
print("dict input:")
tc = adapter._parse_tool_call(
    {
        "type": "tool_use",
        "id": "t1",
        "name": "calc",
        "input": {"expression": "2+2"},
    }
)
print(f"  → {tc}")

print("\nstr input (some providers send JSON-as-string):")
tc = adapter._parse_tool_call(
    {
        "type": "tool_use",
        "id": "t1",
        "name": "calc",
        "input": '{"expression": "2+2"}',
    }
)
print(f"  → {tc}")

print("\nbad JSON string raises ArcLLMParseError:")
try:
    adapter._parse_tool_call(
        {
            "type": "tool_use",
            "id": "t1",
            "name": "calc",
            "input": "not valid json {{{",
        }
    )
except ArcLLMParseError as e:
    print(f"  caught: {type(e).__name__}")
    print(f"  raw_string preserved: {e.raw_string!r}")

print("\nunexpected type also raises:")
try:
    adapter._parse_tool_call(
        {
            "type": "tool_use",
            "id": "t1",
            "name": "calc",
            "input": 12345,
        }
    )
except ArcLLMParseError as e:
    print(f"  caught: {type(e).__name__}")

### Driving a two-turn loop

Patch `httpx.AsyncClient.post` to return turn 1's JSON, then turn 2's.
The harness handles the tool execution between calls.

In [ ]:
get_weather = Tool(
    name="get_weather",
    description="Get current weather for a city",
    parameters={
        "type": "object",
        "properties": {
            "city": {"type": "string"},
            "units": {"type": "string", "enum": ["c", "f"]},
        },
        "required": ["city"],
    },
)


async def run_two_turn_loop() -> None:
    adapter = AnthropicAdapter(fake_config, FAKE_MODEL)

    turn1 = make_anthropic_response(
        [
            {
                "type": "tool_use",
                "id": "toolu_round_1",
                "name": "get_weather",
                "input": {"city": "Austin", "units": "f"},
            }
        ],
        stop_reason="tool_use",
    )
    turn2 = make_anthropic_response(
        [{"type": "text", "text": "Austin is 75 °F and sunny."}],
        stop_reason="end_turn",
    )

    # patch.object is more precise than monkeypatching the module
    with patch.object(adapter._client, "post", fake_post(turn1)):
        messages: list[Message] = [
            Message(role="user", content="What's the weather in Austin?"),
        ]
        first = await adapter.invoke(messages, tools=[get_weather])
        print(f"turn 1: stop_reason={first.stop_reason}, tool={first.tool_calls[0].name}")

    # Harness executes the tool. Append assistant + tool result to history.
    tc = first.tool_calls[0]
    messages.append(
        Message(
            role="assistant",
            content=[ToolUseBlock(id=tc.id, name=tc.name, arguments=tc.arguments)],
        )
    )
    messages.append(
        Message(
            role="tool",
            content=[ToolResultBlock(tool_use_id=tc.id, content="75F sunny")],
        )
    )

    with patch.object(adapter._client, "post", fake_post(turn2)):
        final = await adapter.invoke(messages, tools=[get_weather])
        print(f"turn 2: stop_reason={final.stop_reason}, content={final.content!r}")

    await adapter.close()


await run_two_turn_loop()

## 6. Streaming events on the wire

`AnthropicAdapter.invoke` returns a single `LLMResponse` after the
full body is in. ArcLLM does not currently expose chunk-by-chunk
streaming through the adapter API — accumulation happens at the
provider layer.

Here are the SSE chunk types Anthropic sends when `stream=true` is
set on the request, so you know what's happening underneath. The
adapter's job, were it to consume them, would be to fold these into
the same `LLMResponse` shape `_parse_response` already produces.

In [ ]:
# Anthropic Messages API SSE event types — reference shapes only.
# These are what arrive on the wire when stream=true; we do not stream
# in invoke() today, so this is illustrative.
streaming_events = [
    {
        "type": "message_start",
        "message": {
            "id": "msg_x",
            "model": FAKE_MODEL,
            "usage": {"input_tokens": 12, "output_tokens": 0},
        },
    },
    {"type": "content_block_start", "index": 0, "content_block": {"type": "text", "text": ""}},
    {"type": "content_block_delta", "index": 0, "delta": {"type": "text_delta", "text": "Austin"}},
    {"type": "content_block_delta", "index": 0, "delta": {"type": "text_delta", "text": " is 75"}},
    {"type": "content_block_delta", "index": 0, "delta": {"type": "text_delta", "text": " °F."}},
    {"type": "content_block_stop", "index": 0},
    {"type": "message_delta", "delta": {"stop_reason": "end_turn"}, "usage": {"output_tokens": 9}},
    {"type": "message_stop"},
]
for ev in streaming_events:
    print(ev["type"])

### Folding deltas back into a final response

Even though `invoke` doesn't stream today, the *parse* step is the
same shape — text deltas concatenate, `tool_use` deltas accumulate by
index, then `_parse_response` would run on the assembled body.

This little simulator shows the pattern.

In [ ]:
def fold_text_deltas(events: list[dict[str, Any]]) -> dict[str, Any]:
    """Tiny illustrative folder for text-only stream events."""
    parts: list[str] = []
    stop_reason = "end_turn"
    usage = {"input_tokens": 0, "output_tokens": 0}
    for ev in events:
        if ev["type"] == "content_block_delta":
            delta = ev["delta"]
            if delta.get("type") == "text_delta":
                parts.append(delta["text"])
        elif ev["type"] == "message_start":
            usage["input_tokens"] = ev["message"]["usage"]["input_tokens"]
        elif ev["type"] == "message_delta":
            stop_reason = ev["delta"].get("stop_reason", stop_reason)
            usage["output_tokens"] = ev["usage"]["output_tokens"]
    return {
        "id": "msg_synth",
        "type": "message",
        "role": "assistant",
        "model": FAKE_MODEL,
        "content": [{"type": "text", "text": "".join(parts)}],
        "stop_reason": stop_reason,
        "usage": usage,
    }


synthetic = fold_text_deltas(streaming_events)
resp = adapter._parse_response(synthetic)
print(f"folded content     = {resp.content!r}")
print(f"folded stop_reason = {resp.stop_reason}")
print(f"folded usage       = {resp.usage}")

The point: even if you wire in streaming later, the parser is
already the right shape — you only need to assemble the final
`content` array from deltas.

## 7. Thinking models — `supports_thinking` metadata

Claude's extended-thinking models emit a `thinking` content block
alongside `text`. ArcLLM stores the raw chain-of-thought on
`LLMResponse.thinking` so callers can ignore it (default), display it
(developer mode), or audit it (compliance).

Capability discovery is via the `supports_thinking` field on each
model in the provider TOML — the registry exposes this so callers can
gate behavior without sniffing model name strings.

In [ ]:
# claude-sonnet-4-6 is the current default and supports thinking
real_config = load_provider_config("anthropic")
print(f"default model:        {real_config.provider.default_model}")
print()
print(f"{'model':40s}  thinking  context")
print("-" * 70)
for name, meta in real_config.models.items():
    print(f"{name:40s}  {meta.supports_thinking!s:8s}  {meta.context_window}")

In [ ]:
# Parse a response that includes a thinking block — note the separate field
raw = make_anthropic_response(
    [
        {
            "type": "thinking",
            "thinking": "Let me reason step by step. 7 * 6 = 42, and 42 / 2 = 21...",
        },
        {"type": "text", "text": "The answer is 21."},
    ]
)
resp = adapter._parse_response(raw)
print(f"content   = {resp.content!r}")
print(f"thinking  = {resp.thinking!r}")
print()
print("'thinking' is on its own field — text content is unaffected.")

In [ ]:
# Multiple thinking blocks concatenate the same way text blocks do
raw = make_anthropic_response(
    [
        {"type": "thinking", "thinking": "First, identify the constraint."},
        {"type": "thinking", "thinking": "Second, apply the formula."},
        {"type": "text", "text": "Answer: 42."},
    ]
)
resp = adapter._parse_response(raw)
print(f"thinking has {resp.thinking.count(chr(10)) + 1} parts:")
print(resp.thinking)

### Gating thinking-only behavior in caller code

A common pattern: only enable a "show reasoning" UI when the bound
model advertises `supports_thinking=True`. The metadata makes this
explicit instead of hard-coding model name regexes.

In [ ]:
def thinking_enabled(provider_config: ProviderConfig, model: str) -> bool:
    """True if the bound model is a Claude thinking variant."""
    meta = provider_config.models.get(model)
    return bool(meta and meta.supports_thinking)


for m in ["claude-sonnet-4-6", "claude-opus-4-6", "claude-haiku-4-5-20251001"]:
    print(f"  {m:40s} → thinking={thinking_enabled(real_config, m)}")

## 8. Errors

When the API returns a non-200, `invoke()` raises `ArcLLMAPIError`
with `status_code`, `body`, `provider`, and an optional `retry_after`
parsed from the `Retry-After` response header. This is the contract
the retry/circuit-breaker modules rely on for smart decisions.

In [ ]:
async def call_with_status(
    status: int, body_text: str = "{}", retry_after: str | None = None
) -> ArcLLMAPIError:
    adapter = AnthropicAdapter(fake_config, FAKE_MODEL)
    headers = {"retry-after": retry_after} if retry_after else {}
    response = httpx.Response(
        status,
        text=body_text,
        headers=headers,
        request=httpx.Request("POST", "https://api.anthropic.com/v1/messages"),
    )
    with patch.object(adapter._client, "post", AsyncMock(return_value=response)):
        try:
            await adapter.invoke([Message(role="user", content="hi")])
        except ArcLLMAPIError as e:
            return e
        finally:
            await adapter.close()
    raise AssertionError("expected ArcLLMAPIError")


# 401 — auth failure (don't retry)
err = await call_with_status(401, '{"error":{"message":"invalid x-api-key"}}')
print(f"401 → status_code={err.status_code}, retry_after={err.retry_after}")
print(f"     str(err) = {str(err)[:80]}…")

In [ ]:
# 429 — rate limited, with Retry-After header parsed as seconds
err = await call_with_status(
    429,
    '{"error":{"type":"rate_limit_error","message":"Number of requests..."}}',
    retry_after="12",
)
print(f"429 → retry_after={err.retry_after} seconds")
print(f"     provider={err.provider}")

In [ ]:
# 400 with content_filter — Anthropic surfaces filter rejections as
# regular API errors; the body explains what was filtered.
err = await call_with_status(
    400,
    '{"error":{"type":"invalid_request_error","message":"prompt failed safety classifier"}}',
)
print(f"400 → status_code={err.status_code}")
print(f"     body={err.body[:80]}…")

In [ ]:
# Smart retry decision — the retry module uses status_code to decide
def should_retry(err: ArcLLMAPIError) -> str:
    if err.status_code == 429:
        return f"retry after {err.retry_after}s"
    if err.status_code >= 500:
        return "retry with backoff"
    if err.status_code in (401, 403):
        return "do not retry — credential issue"
    if err.status_code == 400:
        return "do not retry — caller bug"
    return "do not retry"


for code, body in [
    (429, '{"error":"rate"}'),
    (500, '{"error":"oops"}'),
    (401, '{"error":"key"}'),
    (400, '{"error":"bad"}'),
]:
    e = ArcLLMAPIError(
        status_code=code,
        body=body,
        provider="anthropic",
        retry_after=15.0 if code == 429 else None,
    )
    print(f"  {code} → {should_retry(e)}")

## 9. (live) one real call gated on ANTHROPIC_API_KEY

A single live call against the real Anthropic API, gated on
`HAS_LIVE_KEY`. If you have `ANTHROPIC_API_KEY` exported, this runs
end-to-end through the real adapter, the real config, and a real
network request. Otherwise it cleanly skips.

In [ ]:
if not HAS_LIVE_KEY:
    print("skip — set ANTHROPIC_API_KEY")
    raise SystemExit

real_config = load_provider_config("anthropic")
real_model = real_config.provider.default_model


async def live_call() -> None:
    async with AnthropicAdapter(real_config, real_model) as adapter:
        resp = await adapter.invoke(
            [Message(role="user", content="Reply with exactly the word: OK")],
            max_tokens=10,
            temperature=0.0,
        )
        print(f"model        = {resp.model}")
        print(f"content      = {resp.content!r}")
        print(f"stop_reason  = {resp.stop_reason}")
        print(f"input_tokens = {resp.usage.input_tokens}")
        print(f"output_tokens= {resp.usage.output_tokens}")


await live_call()

## 10. Summary

`AnthropicAdapter` is a small, sharply-scoped class that translates
between ArcLLM's universal types and Anthropic's Messages API. Every
method has one job; every quirk lives in exactly one place.

What we covered:

- **Adapter contract** — `invoke`, `validate_config`, `close`. Fail-fast
  key resolution at init via `BaseAdapter` (env var or vault-resolved
  override).
- **Request building** — `_build_headers`, `_extract_system`,
  `_format_content_block`, `_format_message`, `_format_tool`,
  `_build_request_body`. Each is a pure function on already-validated
  Pydantic input.
- **Response parsing** — `_parse_response`, `_parse_tool_call`,
  `_parse_usage`, `_map_stop_reason`. Text, tool_use, and thinking
  blocks fold into `LLMResponse.content / .tool_calls / .thinking`.
- **Tool calls** — round-trip with mocked HTTP. `_parse_arguments`
  handles both dict and JSON-string inputs; garbage raises
  `ArcLLMParseError` with the original string preserved.
- **Streaming** — Anthropic SSE chunk types are illustrated; today
  `invoke` returns the assembled response, but the parser is already
  shaped to fold deltas if streaming is wired in later.
- **Thinking models** — `supports_thinking` metadata in the model TOML
  is the canonical capability check; `claude-sonnet-4-6` and
  `claude-opus-4-6` are the current defaults.
- **Errors** — `ArcLLMAPIError(status_code, body, provider, retry_after)`
  carries enough information for the retry / circuit-breaker /
  fallback modules to make smart decisions.

API surface exercised in this notebook:

- `AnthropicAdapter` — public class, all private methods walked
- `BaseAdapter` — inherited; `validate_config`, `close`,
  `_parse_arguments`, `_resolve_defaults`, `_parse_retry_after`
- `ProviderConfig`, `ProviderSettings`, `ModelMetadata` — fixture and
  real config
- `Message`, `TextBlock`, `ImageBlock`, `ToolUseBlock`,
  `ToolResultBlock`, `Tool` — every input shape
- `LLMResponse`, `Usage`, `StopReason` — every output field
- `ArcLLMAPIError`, `ArcLLMParseError`, `ArcLLMConfigError` — error
  contract
- `load_provider_config` — packaged Anthropic catalog

**Next:** see `arcllm/13-open-providers` for the full adapter catalog
and capability matrix; `arcllm/05-openai-adapter` for the equivalent
OpenAI deep-dive; `arcllm/07-module-system` for how the modules
(retry, fallback, circuit_breaker, queue) consume `ArcLLMAPIError`.